In [1]:
from requests import Session
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd

http = Session()
http.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=5,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods={"HEAD", "GET", "OPTIONS"},
        )
    ),
)

# Define the search query and filters
query = "covid vaccination"
fields = "paperId,title,abstract,year,fieldsOfStudy"
limit = 10  # Fetch up to 1000 papers in one request
publication_types = "JournalArticle,Review"  # Filter by publication types
fields_of_study = "Medicine,Public Health"  # Filter by fields of study
year = "2020-2023"  # Papers published between 2020 and 2023

results = pd.DataFrame()

response = http.get(
    "https://api.semanticscholar.org/graph/v1/paper/search/bulk",
    params={
        "query": query,
        "fields": fields,
        "limit": limit,
        "publicationTypes": publication_types,
        "fieldsOfStudy": fields_of_study,
        "year": year,
    },
)

response.raise_for_status()  # Ensures we stop if there's an error
data = response.json()


# Output the fetched data
print(f"Total matches: {data.get('total')}")
results = pd.DataFrame(data.get("data", None))

# Handle continuation token if there are more results to fetch
if "token" in data:
    print(f"Continuation Token: {data['token']}")

results

Total matches: 34978
Continuation Token: PCOKWVSKJJGM4TWNJNI3EUSQJIWVNUSRKBFEYK2JFUBHFI4VBTGI2DCMRWJJEU4TZRJJHDITZUGSYURMBQGZGTESFUJJGDMSFQWREDIMBRJY3UZMSRRGWAPWUAKH4


,paperId,title,year,openAccessPdf,fieldsOfStudy,authors,abstract
0,0001d8aa78fe5890cd3643fbc7182056d092929c,A framework for pandemic compliant higher educ...,2021,{'url': 'https://link.springer.com/content/pdf...,[Medicine],"[{'authorId': '113657943', 'name': 'Saleh Baja...","Even after 13 months, our world is still battl..."
1,000283ab7417012f31ea3d6d6cafba43ee8ea189,New onset of psoriasis following COVID‐19 vacc...,2022,"{'url': 'https://doi.org/10.1111/dth.15590', '...",[Medicine],"[{'authorId': '2072581921', 'name': 'T. Tran'}...",The cutaneous side effects of COVID‐19 vaccine...
2,0004d5f4d1fc2cd60d299494f1f97384e4614019,Autoimmune inflammatory reactions triggered by...,2023,{'url': 'https://www.tandfonline.com/doi/pdf/1...,[Medicine],"[{'authorId': '2390466173', 'name': 'Panagis P...",Abstract As a result of the spread of SARS-CoV...
3,0005dce282755bf46bf44d1c492133d569cf142e,Barriers and Facilitators Affecting the Uptake...,2023,{'url': 'https://doi.org/10.1177/2377960823115...,[Medicine],"[{'authorId': '72628938', 'name': 'D. Ashipala...",Aim Vaccinations remain one of the most effect...
4,000617d8b5acd6383554e5d04228463c81cd9524,Myofibrillar myopathy presenting with an inclu...,2023,"{'url': '', 'status': 'CLOSED', 'license': Non...",[Medicine],"[{'authorId': '2058274818', 'name': 'F. Diaz'}...","1. Garg RK, Paliwal VK. Spectrum of neurologic..."
...,...,...,...,...,...,...,...
995,071c1878d8b20f72c26f9f17c56d1c47431dab67,Prevention of SARS-CoV-2 Infection: A Liposoma...,2021,"{'url': '', 'status': None, 'license': 'CCBYNC...",[Medicine],"[{'authorId': '50174461', 'name': 'F. Hosseini...",© 2021 International Journal of Preventive Med...
996,071c5094d8fd52e5fbe4f006021732b3df876b43,Health Promotion Strategy To Increase Covid-19...,2023,{'url': 'https://ijhsrd.com/index.php/ijhsrd/a...,None,"[{'authorId': '26937928', 'name': 'Tasnim Tasn...",Background: Covid-19 vaccination coverage is l...
997,071d021b0229fccd522ace1cbc7e4c6fd4cc1397,The immune response of SARS-CoV-2 vaccine in p...,2023,{'url': 'https://balimedicaljournal.org/index....,None,"[{'authorId': '2150772867', 'name': 'Nur Arafa...",Link of Video Abstract: https://youtu.be/rXb-E...
998,071e6e598bb001874f18dc4afa58d249a872d9c2,Anaphylactic Reactions to COVID-19 Vaccine,2021,{'url': 'https://doi.org/10.26440/ihrj/0501.04...,None,"[{'authorId': '80246374', 'name': 'Sachleen Ka...",The SARS-CoV-2 coronavirus responsible for the...


In [ ]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

database = os.getenv("PGDATABASE")
username = os.getenv("PGUSER")
password = os.getenv("PGPASSWORD")
host = os.getenv("PGHOST")
port = os.getenv("PGPORT")


engine = create_engine(
    url=f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

results.to_sql("papery.test_table", engine=engine)

In [ ]:
# import time
# import requests
# from requests.exceptions import RequestException

# API_KEY = "your_api_key_here"
# HEADERS = {"x-api-key": API_KEY}
# URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

# params = {
#     "query": "machine learning",
#     "fields": "paperId,title,year",
#     "limit": 1000,
# }

# MAX_RETRIES = 5
# BASE_DELAY = 1.0      # seconds
# MAX_DELAY = 30.0

# all_results = []

# while True:
#     for attempt in range(MAX_RETRIES):
#         try:
#             response = requests.get(URL, params=params, headers=HEADERS, timeout=30)

#             # Explicit handling of rate limits
#             if response.status_code == 429:
#                 raise RequestException("Rate limited")

#             response.raise_for_status()
#             data = response.json()
#             break

#         except RequestException as e:
#             if attempt == MAX_RETRIES - 1:
#                 raise

#             delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
#             time.sleep(delay)

#     # Process successful page
#     all_results.extend(data.get("data", []))

#     token = data.get("token")
#     if not token:
#         break

#     params["token"] = token

# print(f"Fetched {len(all_results)} results")

HTTPError: 403 Client Error: Forbidden for url: https://api.semanticscholar.org/graph/v1/paper/search/bulk?query=machine+learning&fields=paperId%2Ctitle%2Cyear&limit=1000